# 🧹 Lab 3 — Reset / Cleanup

Undo everything **Lab 3** created so you can run it again — great for practicing or re-demoing. **Safe to run anytime.**

1. 🗑️ deletes your app (compute + service principal + Lakebase binding)
2. 🧹 empties the operational tables the app wrote to (`CLEAR_DATA`)
3. ↩️ restores `app.yaml` + `db.py` to their starting state (`RESET_CODE`)

> ⚠️ Does **not** touch your Lab 1 / Lab 2 data — only what *Lab 3* added.

Nothing to configure — just **Run all**.

Install helpers, then restart Python (run the next cell right after).

In [ ]:
%pip install -U "databricks-sdk>=0.118.0" "psycopg[binary]>=3.1" -q

In [ ]:
dbutils.library.restartPython()

In [ ]:
import re, pathlib
from databricks.sdk import WorkspaceClient
import psycopg

# ── What to reset (flip as you like) ─────────────────────────────────────────
CLEAR_DATA = True    # empty the operational tables the app writes to
RESET_CODE = True    # put app.yaml + db.py back to their starting state (redo the exercise)
# ─────────────────────────────────────────────────────────────────────────────

w = WorkspaceClient()
user = w.current_user.me().user_name
slug = re.sub(r"[^a-z0-9]", "", user.split("@")[0].lower())
APP = ("lb-workshop-" + slug)[:30].rstrip("-")

# Find the app automatically — search from this notebook's folder upward for bundle/src/app.
_dir = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().rsplit("/", 1)[0]
REPO = None
for _ in range(5):
    _cand = _dir if _dir.startswith("/Workspace") else "/Workspace" + _dir
    if pathlib.Path(f"{_cand}/bundle/src/app").exists():
        REPO = _cand
        break
    if _dir.count("/") <= 1:
        break
    _dir = _dir.rsplit("/", 1)[0]
if REPO is None:
    print("⚠️ Couldn't find bundle/src/app near this notebook — skipping the code-reset (RESET_CODE).")
APP_SRC = f"{REPO}/bundle/src/app" if REPO else None

# 1) Delete the app — its compute, service principal and Lakebase binding all go with it.
try:
    w.apps.delete(name=APP)
    print(f"🗑️  deleted app: {APP}")
except Exception as e:
    print(f"app {APP}: {str(e).splitlines()[0]} (already gone?)")

# 2) Empty the tables the app wrote into (so 'Recent actions' starts fresh).
if CLEAR_DATA:
    BRANCH = f"projects/lakebase-ws-{slug}-1/branches/production"
    ENDPOINT = f"{BRANCH}/endpoints/primary"
    host = next(e for e in w.postgres.list_endpoints(parent=BRANCH)
                if e.name == ENDPOINT).status.hosts.host
    with psycopg.connect(host=host, port=5432, dbname="databricks_postgres", user=user,
                         password=w.postgres.generate_database_credential(ENDPOINT).token,
                         sslmode="require", autocommit=True) as c:
        for t in ("maintenance_actions", "work_orders", "quality_checks", "operator_notes"):
            try:
                c.execute(f"TRUNCATE public.{t} RESTART IDENTITY")
                print(f"🧹 cleared public.{t}")
            except Exception as e:
                print(f"{t}: {str(e).splitlines()[0]}")

# 3) Restore the app code to its starting point (undo the Step 1 config + the write-back).
if RESET_CODE and REPO:
    y = pathlib.Path(f"{APP_SRC}/app.yaml"); t = y.read_text()
    t = re.sub(r'(- name: ENDPOINT_NAME\n\s*value: )"[^"]*"', r'\g<1>"REPLACE_ME"', t)
    t = re.sub(r'(- name: PGHOST\n\s*value: )"[^"]*"', r'\g<1>"REPLACE_ME"', t)
    y.write_text(t)

    impl = '''    with conn.cursor() as cur:
        cur.execute(
            """INSERT INTO maintenance_actions
                   (machine_id, ticket_id, action_type, description, performed_by, status, completed_at)
               VALUES (%s, %s, %s, %s, %s, %s, CASE WHEN %s = 'completed' THEN now() END)""",
            (machine_id, ticket_id, action_type, description, performed_by, status, status))
    conn.commit()'''
    stub = '    raise NotImplementedError("Not implemented yet — see Lab 3, Step 4.")'
    d = pathlib.Path(f"{APP_SRC}/db.py"); s = d.read_text()
    if impl in s:
        d.write_text(s.replace(impl, stub)); print("↩️  app.yaml + db.py restored to starting state")
    elif stub in s:
        print("↩️  app.yaml restored; db.py already at starting state")
    else:
        print("↩️  app.yaml restored; db.py was edited differently — run "
              "`git checkout bundle/src/app/db.py` to reset it")

print("\n✅ Reset complete — you can run Lab 3 again from the top.")